# Analyze confidence scores vs correctness

This notebook helps you answer questions like:
- *Are my high-confidence detections usually correct?*
- *What score threshold gives a good precision/recall tradeoff?*

It loads a `result.pkl` produced by `tools/test.py`, matches detections to ground-truth (GT) using an IoU threshold, and then plots:
- score histograms for **TP** vs **FP**
- precision/recall and F1 vs score threshold
- optional calibration-style plot (TP-rate per score bin)

Note on practice: *tuning a threshold on the test set is bad practice* (it overfits). Use **validation** (`val` / `val_small`) to pick thresholds, then report final metrics once on the true test set.

In [ ]:
import os
import pickle
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# If you run this notebook from outside /OpenPCDet, ensure the project root is on sys.path
import sys
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'process_tools' else os.path.abspath(os.getcwd())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Rotated BEV IoU (CPU) via compiled ops (preferred for correctness matching)
from pcdet.ops.iou3d_nms import iou3d_nms_utils

plt.rcParams['figure.figsize'] = (10, 4)
pd.set_option('display.max_columns', 200)

In [ ]:
# --- Paths (edit these) ---
RESULT_PKL = '/OpenPCDet/output/OpenPCDet/tools/cfgs/kitti_models/second_erod/default/eval/epoch_7862/val_small/default/result.pkl'
INFOS_PKL  = '/OpenPCDet/erod/custom_infos_val.pkl'

# --- Matching / analysis settings ---
CLASS_NAME = 'Pedestrian'
IOU_THR = 0.1          # e.g. KITTI Car @0.70; Ped/Cyc often use 0.50
SCORE_GRID = np.linspace(0.0, 1.0, 101)

assert os.path.exists(RESULT_PKL), f'Missing: {RESULT_PKL}'
assert os.path.exists(INFOS_PKL),  f'Missing: {INFOS_PKL}'
print('OK:', RESULT_PKL)
print('OK:', INFOS_PKL)

In [ ]:
def _to_str(x) -> str:
    # handles numpy scalar strings
    try:
        return str(x.item())
    except Exception:
        return str(x)

def load_result_and_infos(result_pkl: str, infos_pkl: str):
    with open(result_pkl, 'rb') as f:
        dets = pickle.load(f)
    with open(infos_pkl, 'rb') as f:
        infos = pickle.load(f)

    infos_by_id = {}
    for info in infos:
        idx = info.get('point_cloud', {}).get('lidar_idx', None)
        if idx is None:
            continue
        infos_by_id[_to_str(idx)] = info

    # normalize det frame_id to string
    for det in dets:
        det['frame_id'] = _to_str(det.get('frame_id', ''))

    return dets, infos_by_id

dets, infos_by_id = load_result_and_infos(RESULT_PKL, INFOS_PKL)
print('detections frames:', len(dets))
print('infos frames:', len(infos_by_id))
print('example det keys:', sorted(dets[0].keys()))
print('example info keys:', sorted(infos_by_id[dets[0]['frame_id']].keys()))

In [ ]:
def classify_detections_for_class(dets, infos_by_id, class_name: str, iou_thr: float):
    records = []
    total_gt = 0
    missing_infos = 0

    for det in dets:
        frame_id = det['frame_id']
        info = infos_by_id.get(frame_id)
        if info is None:
            missing_infos += 1
            continue

        gt_annos = info.get('annos', {})
        gt_boxes = np.asarray(gt_annos.get('gt_boxes_lidar', np.zeros((0, 7))), dtype=np.float32)
        gt_names = np.asarray(gt_annos.get('name', []), dtype=str)

        # IMPORTANT: infos may store yaw in degrees; CustomDataset converts at runtime.
        # Mirror that behavior here so IoUs/TP labels match what evaluation uses.
        if gt_boxes.size > 0 and gt_boxes.shape[1] >= 7:
            max_abs = float(np.max(np.abs(gt_boxes[:, 6])))
            if max_abs > (2 * np.pi + 1.0):
                gt_boxes = gt_boxes.copy()
                gt_boxes[:, 6] = np.deg2rad(gt_boxes[:, 6])

        dt_boxes = np.asarray(det.get('boxes_lidar', np.zeros((0, 7))), dtype=np.float32)
        dt_scores = np.asarray(det.get('score', np.zeros((0,))), dtype=np.float32)
        dt_names = np.asarray(det.get('name', []), dtype=str)

        gt_mask = (gt_names == class_name)
        dt_mask = (dt_names == class_name)

        gt_boxes_c = gt_boxes[gt_mask]
        dt_boxes_c = dt_boxes[dt_mask]
        dt_scores_c = dt_scores[dt_mask]

        total_gt += int(gt_boxes_c.shape[0])

        if dt_boxes_c.shape[0] == 0:
            continue

        if gt_boxes_c.shape[0] == 0:
            # all detections are FP
            for s in dt_scores_c:
                records.append({
                    'frame_id': frame_id,
                    'class': class_name,
                    'score': float(s),
                    'is_tp': False,
                    'best_iou': 0.0
                })
            continue

        # Rotated BEV IoU matrix: [N_dt, N_gt]
        ious = iou3d_nms_utils.boxes_bev_iou_cpu(dt_boxes_c, gt_boxes_c)

        order = np.argsort(-dt_scores_c)
        gt_used = np.zeros((gt_boxes_c.shape[0],), dtype=bool)

        for di in order:
            best_gi = int(np.argmax(ious[di]))
            best_iou = float(ious[di, best_gi])
            is_tp = (best_iou >= iou_thr) and (not gt_used[best_gi])
            if is_tp:
                gt_used[best_gi] = True

            records.append({
                'frame_id': frame_id,
                'class': class_name,
                'score': float(dt_scores_c[di]),
                'is_tp': bool(is_tp),
                'best_iou': best_iou
            })

    if missing_infos:
        print(f'Warning: {missing_infos} frames in result.pkl not found in infos (frame_id mismatch).')

    df = pd.DataFrame.from_records(records)
    return df, total_gt


df, total_gt = classify_detections_for_class(dets, infos_by_id, CLASS_NAME, IOU_THR)
print('detections for class:', len(df))
print('total GT for class:', total_gt)
df.head()


In [ ]:
print(df.to_string())

In [ ]:
# --- Error attribution: translation vs rotation for matched detections ---
# This cell re-matches DT->GT and computes residuals, then estimates how much IoU would
# improve if we "fixed" translation (x,y) or rotation (yaw) while keeping other params.

def _wrap_to_pi(a: np.ndarray) -> np.ndarray:
    return (a + np.pi) % (2 * np.pi) - np.pi

def _maybe_convert_gt_yaw_to_rad(gt_boxes: np.ndarray) -> np.ndarray:
    """Infos may store yaw in degrees; CustomDataset converts at runtime.
    This helper mimics that conversion so notebook metrics match eval behavior.
    """
    if gt_boxes.size == 0 or gt_boxes.shape[1] < 7:
        return gt_boxes
    gt_boxes = gt_boxes.copy()
    max_abs = float(np.max(np.abs(gt_boxes[:, 6])))
    if max_abs > (2 * np.pi + 1.0):
        gt_boxes[:, 6] = np.deg2rad(gt_boxes[:, 6])
    return gt_boxes

def build_match_df(dets, infos_by_id, class_name: str, iou_thr: float):
    rows = []
    for det in dets:
        frame_id = det['frame_id']
        info = infos_by_id.get(frame_id)
        if info is None:
            continue

        gt_annos = info.get('annos', {})
        gt_boxes = np.asarray(gt_annos.get('gt_boxes_lidar', np.zeros((0, 7))), dtype=np.float32)
        gt_names = np.asarray(gt_annos.get('name', []), dtype=str)
        gt_mask = (gt_names == class_name)
        gt_boxes_c = _maybe_convert_gt_yaw_to_rad(gt_boxes[gt_mask])

        dt_boxes = np.asarray(det.get('boxes_lidar', np.zeros((0, 7))), dtype=np.float32)
        dt_scores = np.asarray(det.get('score', np.zeros((0,))), dtype=np.float32)
        dt_names = np.asarray(det.get('name', []), dtype=str)
        dt_mask = (dt_names == class_name)
        dt_boxes_c = dt_boxes[dt_mask]
        dt_scores_c = dt_scores[dt_mask]

        if dt_boxes_c.shape[0] == 0:
            continue

        if gt_boxes_c.shape[0] == 0:
            # No GT => all FP, no residuals
            for s, box in zip(dt_scores_c, dt_boxes_c):
                rows.append({
                    'frame_id': frame_id,
                    'class': class_name,
                    'score': float(s),
                    'is_tp': False,
                    'best_iou': 0.0,
                    'center_dist': np.nan,
                    'abs_dyaw_deg': np.nan,
                    'dx': np.nan,
                    'dy': np.nan,
                })
            continue

        # IoU matrix: [N_dt, N_gt]
        ious = iou3d_nms_utils.boxes_bev_iou_cpu(dt_boxes_c, gt_boxes_c)
        order = np.argsort(-dt_scores_c)
        gt_used = np.zeros((gt_boxes_c.shape[0],), dtype=bool)

        for di in order:
            best_gi = int(np.argmax(ious[di]))
            best_iou = float(ious[di, best_gi])
            is_tp = (best_iou >= iou_thr) and (not gt_used[best_gi])
            if is_tp:
                gt_used[best_gi] = True

            dtb = dt_boxes_c[di]
            gtb = gt_boxes_c[best_gi]
            dxy = dtb[0:2] - gtb[0:2]
            dyaw = float(_wrap_to_pi(np.array([dtb[6] - gtb[6]], dtype=np.float32))[0])

            rows.append({
                'frame_id': frame_id,
                'class': class_name,
                'score': float(dt_scores_c[di]),
                'is_tp': bool(is_tp),
                'best_iou': best_iou,
                'center_dist': float(np.linalg.norm(dxy)),
                'abs_dyaw_deg': float(abs(dyaw) * 180.0 / np.pi),
                'dx': float(dxy[0]),
                'dy': float(dxy[1]),
                # stash boxes for what-if IoU calculations
                'dt_box': dtb.astype(np.float32),
                'gt_box': gtb.astype(np.float32),
            })

    return pd.DataFrame(rows)

match_df = build_match_df(dets, infos_by_id, CLASS_NAME, IOU_THR)
print('match_df rows:', len(match_df), 'TP:', int(match_df.is_tp.sum()), 'FP:', int((~match_df.is_tp).sum()))

# Focus on TPs (these are the ones whose IoU is "capped" around 0.2-0.3 in your printout)
match_tp = match_df[match_df.is_tp].copy()
if len(match_tp) == 0:
    raise RuntimeError('No TPs found for this class at current IOU_THR; lower IOU_THR or change CLASS_NAME.')

print('TP IoU quantiles:', match_tp.best_iou.quantile([0.1, 0.25, 0.5, 0.75, 0.9]).to_dict())
print('TP center_dist(m) quantiles:', match_tp.center_dist.quantile([0.1, 0.25, 0.5, 0.75, 0.9]).to_dict())
print('TP abs_dyaw(deg) quantiles:', match_tp.abs_dyaw_deg.quantile([0.1, 0.25, 0.5, 0.75, 0.9]).to_dict())

def _pair_bev_iou(dt_box: np.ndarray, gt_box: np.ndarray) -> float:
    dt = np.asarray(dt_box, dtype=np.float32).reshape(1, -1)
    gt = np.asarray(gt_box, dtype=np.float32).reshape(1, -1)
    return float(iou3d_nms_utils.boxes_bev_iou_cpu(dt, gt)[0, 0])

# What-if: keep size, but fix yaw and/or xy
orig = []
yaw_fixed = []
xy_fixed = []
xy_yaw_fixed = []

for _, r in match_tp.iterrows():
    dtb = r['dt_box'].copy()
    gtb = r['gt_box'].copy()

    orig.append(_pair_bev_iou(dtb, gtb))

    dt_y = dtb.copy()
    dt_y[6] = gtb[6]
    yaw_fixed.append(_pair_bev_iou(dt_y, gtb))

    dt_xy = dtb.copy()
    dt_xy[0:2] = gtb[0:2]
    xy_fixed.append(_pair_bev_iou(dt_xy, gtb))

    dt_xyy = dtb.copy()
    dt_xyy[0:2] = gtb[0:2]
    dt_xyy[6] = gtb[6]
    xy_yaw_fixed.append(_pair_bev_iou(dt_xyy, gtb))

orig = np.array(orig)
yaw_fixed = np.array(yaw_fixed)
xy_fixed = np.array(xy_fixed)
xy_yaw_fixed = np.array(xy_yaw_fixed)

print('\nMean IoU (orig):', float(orig.mean()))
print('Mean IoU (yaw fixed):', float(yaw_fixed.mean()), 'Δ', float((yaw_fixed - orig).mean()))
print('Mean IoU (xy fixed):', float(xy_fixed.mean()), 'Δ', float((xy_fixed - orig).mean()))
print('Mean IoU (xy+yaw fixed):', float(xy_yaw_fixed.mean()), 'Δ', float((xy_yaw_fixed - orig).mean()))

def bootstrap_ci(delta: np.ndarray, n_boot: int = 2000, seed: int = 0):
    rng = np.random.default_rng(seed)
    n = len(delta)
    means = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        means.append(float(delta[idx].mean()))
    lo, hi = np.quantile(means, [0.025, 0.975])
    return float(lo), float(hi)

print('\nBootstrap 95% CI for mean ΔIoU:')
for name, delta in [
    ('yaw fixed', yaw_fixed - orig),
    ('xy fixed', xy_fixed - orig),
    ('xy+yaw fixed', xy_yaw_fixed - orig),
]:
    lo, hi = bootstrap_ci(delta)
    print(f'- {name}: mean Δ={float(delta.mean()):.4f}, 95% CI=({lo:.4f}, {hi:.4f})')

# Simple rank-correlation (Spearman) between IoU and each error metric for TPs
# (We avoid SciPy; this is enough to see which error most strongly tracks IoU.)

def _spearman_r(x: np.ndarray, y: np.ndarray) -> float:
    x = np.asarray(x)
    y = np.asarray(y)
    xr = x.argsort().argsort().astype(np.float32)
    yr = y.argsort().argsort().astype(np.float32)
    xr = xr - xr.mean()
    yr = yr - yr.mean()
    denom = float(np.sqrt((xr * xr).sum() * (yr * yr).sum()))
    return float((xr * yr).sum() / denom) if denom > 0 else 0.0

r_center = _spearman_r(match_tp.best_iou.values, match_tp.center_dist.values)
r_yaw = _spearman_r(match_tp.best_iou.values, match_tp.abs_dyaw_deg.values)
print('\nSpearman r (IoU vs center_dist):', r_center)
print('Spearman r (IoU vs abs_dyaw_deg):', r_yaw)


In [ ]:
# --- Statistical significance: does IoU track translation error or rotation error more? ---
# We keep this SciPy-free via permutation / sign-flip tests.

import numpy as np

if 'match_tp' not in globals() or len(match_tp) == 0:
    raise RuntimeError('match_tp is missing/empty. Run the previous cell first.')

# Helpers

def spearman_r(x: np.ndarray, y: np.ndarray) -> float:
    x = np.asarray(x)
    y = np.asarray(y)
    # rank with tie-breaking by order; good enough for diagnostics
    xr = x.argsort().argsort().astype(np.float32)
    yr = y.argsort().argsort().astype(np.float32)
    xr = xr - xr.mean()
    yr = yr - yr.mean()
    denom = float(np.sqrt((xr * xr).sum() * (yr * yr).sum()))
    return float((xr * yr).sum() / denom) if denom > 0 else 0.0


def perm_pvalue_spearman(x: np.ndarray, y: np.ndarray, n_perm: int = 5000, seed: int = 0):
    rng = np.random.default_rng(seed)
    x = np.asarray(x)
    y = np.asarray(y)
    obs = spearman_r(x, y)
    more_extreme = 0
    for _ in range(n_perm):
        yp = rng.permutation(y)
        rp = spearman_r(x, yp)
        if abs(rp) >= abs(obs):
            more_extreme += 1
    p = (more_extreme + 1) / (n_perm + 1)
    return obs, float(p)


def signflip_pvalue_mean(delta: np.ndarray, n_perm: int = 20000, seed: int = 0):
    """Paired test for mean(delta) != 0 using random sign-flips."""
    rng = np.random.default_rng(seed)
    delta = np.asarray(delta, dtype=np.float32)
    obs = float(delta.mean())
    n = delta.size
    # vectorized: (n_perm, n)
    signs = rng.choice(np.array([-1.0, 1.0], dtype=np.float32), size=(n_perm, n), replace=True)
    perm_means = (signs * delta.reshape(1, -1)).mean(axis=1)
    p = float((np.sum(np.abs(perm_means) >= abs(obs)) + 1) / (n_perm + 1))
    lo, hi = np.quantile(perm_means, [0.025, 0.975])
    return obs, p, float(lo), float(hi)


def bootstrap_mean_ci(x: np.ndarray, n_boot: int = 5000, seed: int = 0):
    rng = np.random.default_rng(seed)
    x = np.asarray(x, dtype=np.float32)
    n = x.size
    means = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        means.append(float(x[idx].mean()))
    lo, hi = np.quantile(means, [0.025, 0.975])
    return float(np.mean(x)), float(lo), float(hi)


def qstr(x: np.ndarray, qs=(0.1, 0.25, 0.5, 0.75, 0.9)):
    x = np.asarray(x, dtype=np.float32)
    return {q: float(np.quantile(x, q)) for q in qs}


# 1) Correlation: IoU vs each error metric (TPs only)
print(f'TP sample size: {len(match_tp)}')

for metric in ['center_dist', 'abs_dyaw_deg']:
    r, p = perm_pvalue_spearman(match_tp['best_iou'].values, match_tp[metric].values, n_perm=5000, seed=0)
    print(f'Spearman r(IoU vs {metric}) = {r:+.3f}, permutation p≈{p:.4g}')

# 2) What-if deltas: if we could correct xy and/or yaw, how much does IoU rise?
d_yaw = yaw_fixed - orig

d_xy = xy_fixed - orig

d_xyyaw = xy_yaw_fixed - orig

for name, delta in [('yaw fixed', d_yaw), ('xy fixed', d_xy), ('xy+yaw fixed', d_xyyaw)]:
    obs, p, lo, hi = signflip_pvalue_mean(delta, n_perm=20000, seed=0)
    print(f'Mean ΔIoU ({name}): {obs:+.4f} | sign-flip p≈{p:.4g} | null-mean 95% band=({lo:+.4f}, {hi:+.4f})')

# 3) Per-example attribution: which fix helps more?
better_xy = d_xy > d_yaw
better_yaw = d_yaw > d_xy
print('\nPer-example:')
print('Fraction where xy-fix helps more than yaw-fix:', float(np.mean(better_xy)))
print('Fraction where yaw-fix helps more than xy-fix:', float(np.mean(better_yaw)))
print('Fraction where they tie (within 1e-6):', float(np.mean(np.isclose(d_xy, d_yaw, atol=1e-6))))

# 4) Quick “high IoU vs low IoU” group comparison (TPs only)
q = match_tp['best_iou'].quantile([0.25, 0.75]).to_dict()
low = match_tp[match_tp.best_iou <= q[0.25]]
high = match_tp[match_tp.best_iou >= q[0.75]]

print('\nTP IoU quartiles:', q)
for metric in ['center_dist', 'abs_dyaw_deg']:
    low_med = float(np.median(low[metric].values))
    high_med = float(np.median(high[metric].values))
    print(f'{metric}: median(low-IoU)={low_med:.3f} | median(high-IoU)={high_med:.3f} | Δ(high-low)={high_med-low_med:+.3f}')

# 5) Break translation error into dx vs dy (TPs only)
if 'dx' not in match_tp.columns or 'dy' not in match_tp.columns:
    raise RuntimeError('match_tp is missing dx/dy columns. Ensure build_match_df stores dx/dy residuals.')

dx = match_tp['dx'].values.astype(np.float32)
dy = match_tp['dy'].values.astype(np.float32)
abs_dx = np.abs(dx)
abs_dy = np.abs(dy)
delta_abs = abs_dx - abs_dy  # >0 means x-error larger than y-error

print('\n--- dx/dy diagnostics (TPs only) ---')
print('Signed dx quantiles (m):', qstr(dx))
print('Signed dy quantiles (m):', qstr(dy))
print('Abs |dx| quantiles (m):', qstr(abs_dx))
print('Abs |dy| quantiles (m):', qstr(abs_dy))

# Consistent offset check (mean dx/dy significantly != 0?)
dx_mean, dx_ci_lo, dx_ci_hi = bootstrap_mean_ci(dx, n_boot=5000, seed=0)
dy_mean, dy_ci_lo, dy_ci_hi = bootstrap_mean_ci(dy, n_boot=5000, seed=1)
print(f'Mean dx (m): {dx_mean:+.4f} | bootstrap 95% CI=({dx_ci_lo:+.4f}, {dx_ci_hi:+.4f})')
print(f'Mean dy (m): {dy_mean:+.4f} | bootstrap 95% CI=({dy_ci_lo:+.4f}, {dy_ci_hi:+.4f})')

obs, p, _, _ = signflip_pvalue_mean(dx, n_perm=20000, seed=2)
print(f'Sign-flip test mean dx!=0: mean={obs:+.4f}, p≈{p:.4g}')
obs, p, _, _ = signflip_pvalue_mean(dy, n_perm=20000, seed=3)
print(f'Sign-flip test mean dy!=0: mean={obs:+.4f}, p≈{p:.4g}')

# Which axis dominates in magnitude? (tests mean(|dx|-|dy|)!=0)
obs, p, _, _ = signflip_pvalue_mean(delta_abs, n_perm=20000, seed=4)
dom = 'x' if obs > 0 else 'y'
print(f'Mean(|dx|-|dy|)={obs:+.4f} => typical error larger in {dom}-axis | sign-flip p≈{p:.4g}')

# Correlation of IoU with each axis error
for metric_name, arr in [('abs_dx', abs_dx), ('abs_dy', abs_dy), ('abs_dx-abs_dy', delta_abs)]:
    r, p = perm_pvalue_spearman(match_tp['best_iou'].values, arr, n_perm=5000, seed=5)
    print(f'Spearman r(IoU vs {metric_name}) = {r:+.3f}, permutation p≈{p:.4g}')

# If you see a strong nonzero mean dx/dy, that often indicates a constant frame/translation offset.
# If |dx| dominates |dy| consistently, it can indicate a partial axis scaling/offset, or a systematic heading convention issue mapping forward/left.


In [ ]:
# Score distributions: TP vs FP
if len(df) == 0:
    raise RuntimeError('No detections found for this class. Change CLASS_NAME or check RESULT_PKL.')

tp = df[df.is_tp]
fp = df[~df.is_tp]

bins = np.linspace(0, 1, 41)
plt.hist(tp.score, bins=bins, alpha=0.6, label=f'TP (n={len(tp)})')
plt.hist(fp.score, bins=bins, alpha=0.6, label=f'FP (n={len(fp)})')
plt.title(f'{CLASS_NAME} score histogram (IoU≥{IOU_THR})')
plt.xlabel('confidence score')
plt.ylabel('count')
plt.legend()
plt.show()

In [ ]:
def sweep_thresholds(df: pd.DataFrame, total_gt: int, score_grid: np.ndarray):
    out = []
    for thr in score_grid:
        keep = df[df.score >= thr]
        tp = int(keep.is_tp.sum())
        fp = int((~keep.is_tp).sum())
        precision = tp / (tp + fp) if (tp + fp) else 1.0
        recall = tp / total_gt if total_gt else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        out.append({
            'thr': float(thr),
            'tp': tp,
            'fp': fp,
            'precision': float(precision),
            'recall': float(recall),
            'f1': float(f1),
        })
    return pd.DataFrame(out)

sweep = sweep_thresholds(df, total_gt, SCORE_GRID)
best = sweep.iloc[int(np.argmax(sweep.f1))]
best

In [ ]:
# Precision/Recall vs threshold
plt.plot(sweep.thr, sweep.precision, label='precision')
plt.plot(sweep.thr, sweep.recall, label='recall')
plt.plot(sweep.thr, sweep.f1, label='f1')
plt.axvline(best.thr, color='k', linestyle='--', linewidth=1, label='best F1 threshold')
plt.title(f'{CLASS_NAME} threshold sweep (IoU≥{IOU_THR})')
plt.xlabel('score threshold')
plt.ylim(0, 1.05)
plt.grid(True, alpha=0.2)
plt.legend()
plt.show()

print('Best-F1 threshold:', float(best.thr))
print('precision/recall/f1:', float(best.precision), float(best.recall), float(best.f1))
print('tp/fp:', int(best.tp), int(best.fp))

In [ ]:
# Optional: calibration-style plot (TP-rate per score bin)
# This is not a full calibration analysis, but it helps spot if scores are over/under-confident.
n_bins = 10
bins = np.linspace(0, 1, n_bins + 1)
df2 = df.copy()
df2['bin'] = np.digitize(df2.score, bins, right=True)
rows = []
for b in range(1, n_bins + 1):
    part = df2[df2.bin == b]
    if len(part) == 0:
        continue
    rows.append({
        'bin': b,
        'score_mid': float((bins[b-1] + bins[b]) / 2),
        'count': int(len(part)),
        'tp_rate': float(part.is_tp.mean())
    })
cal = pd.DataFrame(rows)

plt.plot(cal.score_mid, cal.tp_rate, marker='o', label='TP-rate per bin')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='ideal (y=x)')
plt.title(f'{CLASS_NAME} score vs TP-rate (IoU≥{IOU_THR})')
plt.xlabel('score (bin midpoint)')
plt.ylabel('TP fraction in bin')
plt.ylim(0, 1.05)
plt.grid(True, alpha=0.2)
plt.legend()
plt.show()

cal

## Interpreting results

- If TP and FP score histograms overlap heavily, thresholding won’t help much; the model isn’t separating correct vs incorrect predictions well.
- If most FPs are low-score and TPs are high-score, a threshold can help a lot.
- AP metrics are mostly *threshold free* (they integrate over all thresholds). Picking a single threshold is mainly for deployment or visualization.

### Good practice
- Pick thresholds on **val** (or `val_small`) only.
- Report final numbers once on the held-out test set.